# Stage 11 — Naked option buying scale-up (218 stocks + 8 indices, 1H)

Same entry signal as the debit spread strategy (`ema_adx_signal.py`, EMA 9/21 + rising ADX, 1H), but `option_buying_trades.py` exits with a trailing stop instead of a fixed 1.5R target - naked buying has no structural cap the way a spread does, so a fixed target would throw away the whole reason to skip the hedge leg.

**4H trend hard filter added after validation** (`apply_4h_trend_filter()`, baked into `option_buying_scale_pipeline.process_instrument()`/`reprocess_instrument()`): only 1H entries that agree with the symbol's own 4H Supertrend direction at that moment survive. Tested on 12,608 real closed trades matched to their 4H trend state: aligned entries won 53.3% vs 47.2% against, mean R 0.166 vs 0.024, median R +0.08 vs -0.46 - the typical against-trend trade was a loser, the typical aligned trade wasn't. This roughly halves signal frequency by construction, so per-instrument trade counts here will be noticeably lower than a pre-filter run.

A sandbox run (cached 1H data, hand-rolled pandas_ta stand-in since the real library isn't installable there) with the filter applied gave 153 qualified instruments (>=15 closed trades, down from 189 pre-filter), 70.6% positive avg R, mean 0.198, median 0.170 - all higher than the pre-filter numbers, at the cost of a thinner per-instrument sample (mean ~19 closed trades vs ~30+ before). That run used whatever history happened to already be cached per symbol (inconsistent windows), not a clean uniform one. **This notebook does the real, clean version**: same `from_date=2023-01-01` window as `14_debit_spread_scale_all_fno.ipynb` (Stage 9), `force_refresh=True` so every symbol gets the same window regardless of what's currently cached.

New: `src/option_buying_scale_pipeline.py` - parallel to `debit_spread_scale_pipeline.py`, not a modification. Same resume/failure-isolation/rate-limit discipline.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import get_fno_stock_list, build_nse_token_lookup, get_index_token
from src.option_buying_scale_pipeline import run_all_instruments, build_comparison, save_comparison

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
stock_list = get_fno_stock_list(kite)
len(stock_list)

224

In [2]:
results = run_all_instruments(
    kite, token_lookup, stock_list,
    from_date=datetime(2023, 1, 1),
    to_date=datetime.now(),
    force_refresh=True
)
results

[1/224] 360ONE — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6529 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_60minute.csv
Saved 42 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/360ONE_trades.csv
[1/224] 360ONE — done (trades=42, win_rate=45.2, avg_r=0.1)
[2/224] ABB — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6529 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABB_60minute.csv
Saved 34 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/ABB_trades.csv
[2/224] ABB — done (trades=34, win_rate=50.0, avg_r=-0.06)
[3/224] ABCAPITAL — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 20

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,pct_trades_trailing_activated,symbol,status,error
0,42.0,42.0,0.0,45.2,0.10,-0.62,"{'STOP': 20, 'TRAIL_STOP': 12, 'TIME': 10}",7.0,42.9,360ONE,SUCCESS,
1,34.0,34.0,0.0,50.0,-0.06,-0.48,"{'STOP': 17, 'TRAIL_STOP': 13, 'TIME': 4}",6.0,47.1,ABB,SUCCESS,
2,44.0,44.0,0.0,50.0,0.04,-0.04,"{'STOP': 21, 'TRAIL_STOP': 20, 'TIME': 3}",6.0,47.7,ABCAPITAL,SUCCESS,
3,36.0,36.0,0.0,63.9,0.45,0.44,"{'TRAIL_STOP': 18, 'STOP': 12, 'TIME': 6}",5.0,61.1,ADANIENSOL,SUCCESS,
4,39.0,39.0,0.0,48.7,0.02,-1.00,"{'STOP': 20, 'TRAIL_STOP': 13, 'TIME': 6}",6.0,46.2,ADANIENT,SUCCESS,
...,...,...,...,...,...,...,...,...,...,...,...,...
219,31.0,31.0,0.0,61.3,0.59,0.28,"{'TRAIL_STOP': 14, 'STOP': 11, 'TIME': 6}",8.0,61.3,VOLTAS,SUCCESS,
220,19.0,19.0,0.0,63.2,0.23,0.19,"{'TRAIL_STOP': 10, 'STOP': 6, 'TIME': 3}",6.0,57.9,WAAREEENER,SUCCESS,
221,41.0,41.0,0.0,56.1,0.14,0.15,"{'TRAIL_STOP': 20, 'STOP': 18, 'TIME': 3}",7.0,53.7,WIPRO,SUCCESS,
222,50.0,50.0,0.0,42.0,0.14,-1.00,"{'STOP': 29, 'TRAIL_STOP': 18, 'TIME': 3}",5.5,42.0,YESBANK,SUCCESS,


## Indices — same 6 as the debit spread run, separate token lookup

In [3]:
from src.option_buying_scale_pipeline import process_instrument
import pandas as pd

indices = [
    {"symbol": "NIFTY", "tradingsymbol": "NIFTY 50", "exchange": "NSE"},
    {"symbol": "BANKNIFTY", "tradingsymbol": "NIFTY BANK", "exchange": "NSE"},
    {"symbol": "SENSEX", "tradingsymbol": "SENSEX", "exchange": "BSE"},
    {"symbol": "NIFTYNXT50", "tradingsymbol": "NIFTY NEXT 50", "exchange": "NSE"},
    {"symbol": "MIDCPNIFTY", "tradingsymbol": "NIFTY MID SELECT", "exchange": "NSE"},
    {"symbol": "FINNIFTY", "tradingsymbol": "NIFTY FIN SERVICE", "exchange": "NSE"},
]

index_results = []
for idx in indices:
    token = get_index_token(kite, idx["tradingsymbol"], idx["exchange"])
    summary = process_instrument(idx["symbol"], token, kite,
                                  from_date=datetime(2023, 1, 1), to_date=datetime.now())
    summary["status"] = "SUCCESS"
    summary["error"] = ""
    index_results.append(summary)
    print(f"{idx['symbol']} — refreshed")

index_results_df = pd.DataFrame(index_results)
index_results_df

Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6589 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 35 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/NIFTY_trades.csv
NIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6589 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_60minute.csv
Saved 35 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/BANKNIFTY_trades.csv
BANKNIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6589 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SENSEX_60minute.csv
Saved 28 trades to /Users/hgmac/Documents/3_tr

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,pct_trades_trailing_activated,symbol,status,error
0,35,35,0,48.6,0.17,-1.00,"{'STOP': 18, 'TRAIL_STOP': 10, 'TIME': 7}",6.0,42.9,NIFTY,SUCCESS,
1,35,35,0,54.3,0.27,0.25,"{'STOP': 15, 'TRAIL_STOP': 14, 'TIME': 6}",7.0,54.3,BANKNIFTY,SUCCESS,
2,28,28,0,57.1,0.23,0.16,"{'TRAIL_STOP': 12, 'STOP': 10, 'TIME': 6}",7.0,50.0,SENSEX,SUCCESS,
3,38,38,0,60.5,0.47,0.16,"{'TRAIL_STOP': 18, 'STOP': 14, 'TIME': 6}",7.0,60.5,NIFTYNXT50,SUCCESS,
4,43,43,0,55.8,0.41,0.12,"{'TRAIL_STOP': 18, 'STOP': 17, 'TIME': 8}",6.0,53.5,MIDCPNIFTY,SUCCESS,
5,37,37,0,43.2,-0.09,-1.00,"{'STOP': 19, 'TRAIL_STOP': 13, 'TIME': 5}",6.0,43.2,FINNIFTY,SUCCESS,


## Aggregate comparison — the actual verdict

In [4]:
all_results = pd.concat([results, index_results_df], ignore_index=True)
comparison = build_comparison(all_results)
save_comparison(comparison)
comparison

Saved option buying comparison of 224 instruments to /Users/hgmac/Documents/3_trading/Quant/exports/OPTION_BUYING_COMPARISON.xlsx


,symbol,total_trades,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,SWIGGY,20.0,20.0,80.0,1.10,0.94,6.0,75.0
1,PFC,28.0,28.0,71.4,0.88,0.72,7.0,71.4
2,RVNL,43.0,43.0,74.4,0.83,0.74,7.0,72.1
3,SHRIRAMFIN,38.0,38.0,68.4,0.81,0.20,7.0,63.2
4,HINDUNILVR,40.0,40.0,65.0,0.79,0.56,9.5,55.0
...,...,...,...,...,...,...,...,...
219,CUB,0.0,0.0,NaN,NaN,NaN,NaN,NaN
220,DATAPATTNS,0.0,0.0,NaN,NaN,NaN,NaN,NaN
221,KARURVYSYA,0.0,0.0,NaN,NaN,NaN,NaN,NaN
222,NAVINFLUOR,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [5]:
# Same >= 15 closed trades floor used everywhere else in this project.
enough_trades = comparison[comparison["closed_trades"] >= 15]

print(f"Instruments processed: {len(comparison)}")
print(f"Instruments with >= 15 closed trades: {len(enough_trades)}")
print(f"Fraction with positive avg_r_achieved: "
      f"{(enough_trades['avg_r_achieved'] > 0).mean() * 100:.1f}%")
print()
print("avg_r_achieved distribution:")
print(enough_trades["avg_r_achieved"].describe())
print()
print("win_rate_pct distribution:")
print(enough_trades["win_rate_pct"].describe())
print()
print("pct_trades_trailing_activated distribution:")
print(enough_trades["pct_trades_trailing_activated"].describe())

Instruments processed: 224
Instruments with >= 15 closed trades: 217
Fraction with positive avg_r_achieved: 80.2%

avg_r_achieved distribution:
count    217.000000
mean       0.200599
std        0.235724
min       -0.310000
25%        0.050000
50%        0.180000
75%        0.310000
max        1.100000
Name: avg_r_achieved, dtype: float64

win_rate_pct distribution:
count    217.000000
mean      54.731797
std        7.990883
min       34.100000
25%       48.700000
50%       54.500000
75%       60.500000
max       80.000000
Name: win_rate_pct, dtype: float64

pct_trades_trailing_activated distribution:
count    217.000000
mean      51.325806
std        8.099999
min       29.300000
25%       45.500000
50%       51.500000
75%       56.400000
max       75.000000
Name: pct_trades_trailing_activated, dtype: float64
